# DINOv2 Partial Fine-Tuning + Confidence Thresholding

**Improvement:** Unfreeze the last few transformer blocks of DINOv2 to adapt high-level features to our domain, while keeping lower-level features frozen.

**Baseline to beat:** DINOv2 frozen + 5-class threshold at 0.75 → Macro F1: 0.8628

---

## 1. Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data

In [ ]:
TRAIN_DIR = './data_train'
TEST_DIR = './data_test'

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)

# 5-class setup (excluding "other")
CLEAN_CLASS_NAMES = [c for c in CLASS_NAMES if c != 'other']
NUM_CLEAN = len(CLEAN_CLASS_NAMES)

print(f'All classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Clean classes ({NUM_CLEAN}): {CLEAN_CLASS_NAMES}')

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16  # smaller batch — we're fine-tuning backbone, uses more GPU memory
NUM_EPOCHS = 30
PATIENCE = 7

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# Full 6-class datasets (needed for threshold evaluation)
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
full_test_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=test_transform)

# 5-class filtered dataset (for training)
other_class_idx = full_train_dataset.class_to_idx['other']
clean_indices = [i for i, (_, label) in enumerate(full_train_dataset.samples) if label != other_class_idx]
clean_subset = Subset(full_train_dataset, clean_indices)

# Remap labels: skip "other" index, map to 0-4
old_to_new = {}
new_idx = 0
for old_idx, cls_name in enumerate(CLASS_NAMES):
    if cls_name != 'other':
        old_to_new[old_idx] = new_idx
        new_idx += 1

class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, label_map):
        self.subset = subset
        self.label_map = label_map
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        return image, self.label_map[label]

clean_train_dataset = RemappedDataset(clean_subset, old_to_new)

# num_workers=0 for RemappedDataset (Windows compatibility)
clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=0, pin_memory=True)
full_test_loader = DataLoader(full_test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

print(f'5-class train: {len(clean_train_dataset)} images, {len(clean_train_loader)} batches')
print(f'6-class test:  {len(full_test_dataset)} images')
print(f'Label mapping: {old_to_new}')

## 3. Model: DINOv2 with Partial Unfreezing

DINOv2 ViT-S has 12 transformer blocks. We'll experiment with unfreezing the last N blocks.

**Why partial unfreezing?**
- Lower blocks learn generic features (edges, textures) — these transfer perfectly, no need to change
- Higher blocks learn semantic features (object composition, scene layout) — these can be tuned for our specific classes
- The difference between `tu_hop` (group gathering) and `ngay_tet` (Tet celebration with a group) depends on subtle high-level cues that domain-adapted features can capture better

In [ ]:
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print(f'DINOv2 ViT-S loaded. Feature dim: {dinov2_backbone.embed_dim}')

# Inspect the backbone structure
print(f'\nNumber of transformer blocks: {len(dinov2_backbone.blocks)}')
print(f'\nBackbone children:')
for name, _ in dinov2_backbone.named_children():
    print(f'  {name}')

In [ ]:
class DINOv2PartialFinetune(nn.Module):
    """
    DINOv2 with last N transformer blocks unfrozen for domain adaptation.
    
    Frozen: patch_embed, pos_embed, cls_token, blocks[0:12-N]
    Trainable: blocks[12-N:12], norm, classifier head
    """
    def __init__(self, backbone, num_classes, unfreeze_last_n=2, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        self.unfreeze_last_n = unfreeze_last_n
        num_blocks = len(backbone.blocks)
        
        # Step 1: Freeze everything
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Step 2: Unfreeze last N transformer blocks
        for i in range(num_blocks - unfreeze_last_n, num_blocks):
            for param in self.backbone.blocks[i].parameters():
                param.requires_grad = True
        
        # Step 3: Unfreeze the final LayerNorm (after blocks)
        for param in self.backbone.norm.parameters():
            param.requires_grad = True
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
        
        # Report
        frozen_blocks = list(range(0, num_blocks - unfreeze_last_n))
        unfrozen_blocks = list(range(num_blocks - unfreeze_last_n, num_blocks))
        print(f'Frozen blocks:   {frozen_blocks}')
        print(f'Unfrozen blocks: {unfrozen_blocks} + norm + classifier')
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
# Compare parameter counts for different unfreeze levels
print('Parameter comparison by unfreeze level:\n')
print(f'{"Unfrozen blocks":>16} | {"Trainable":>12} | {"% of total":>10}')
print('-' * 48)

for n in [0, 1, 2, 3, 4]:
    temp = DINOv2PartialFinetune(dinov2_backbone, NUM_CLEAN, unfreeze_last_n=n)
    total, trainable = count_params(temp)
    print(f'{n:>16} | {trainable:>12,} | {trainable/total*100:>9.2f}%')
    del temp

print(f'\nNote: 0 blocks = fully frozen baseline (head only)')

## 4. Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_5cls(model, loader, criterion, device):
    """Evaluate 5-class model on 5-class data."""
    model.eval()
    running_loss = 0.0; correct = 0; total = 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def predict_with_threshold(model, loader, threshold, clean_class_names, all_class_names, device):
    """
    Run 5-class model on full 6-class test set.
    If max confidence < threshold → predict 'other'.
    Returns predictions mapped back to 6-class indices.
    """
    model.eval()
    other_idx = all_class_names.index('other')
    
    # Map 5-class indices back to 6-class indices
    new_to_old = {}
    clean_i = 0
    for old_i, cls in enumerate(all_class_names):
        if cls != 'other':
            new_to_old[clean_i] = old_i
            clean_i += 1
    
    all_preds_6cls = []
    all_labels_6cls = []
    all_confidences = []
    
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        max_probs, pred_5cls = probs.max(dim=1)
        
        for i in range(len(labels)):
            conf = max_probs[i].item()
            all_confidences.append(conf)
            all_labels_6cls.append(labels[i].item())
            
            if conf < threshold:
                all_preds_6cls.append(other_idx)
            else:
                all_preds_6cls.append(new_to_old[pred_5cls[i].item()])
    
    return np.array(all_preds_6cls), np.array(all_labels_6cls), np.array(all_confidences)

## 5. Experiment: Compare Unfreeze Levels

We'll train models with 0 (baseline), 1, 2, and 3 unfrozen blocks, then compare.

In [ ]:
def run_experiment(unfreeze_n, backbone, clean_loader, test_loader,
                   clean_class_names, all_class_names, device,
                   num_epochs=30, patience=7):
    """
    Train a model with N unfrozen blocks, find optimal threshold, return results.
    """
    print(f'\n{"="*70}')
    print(f'EXPERIMENT: Unfreeze last {unfreeze_n} blocks')
    print(f'{"="*70}')
    
    # Build model
    model = DINOv2PartialFinetune(
        backbone, len(clean_class_names), unfreeze_last_n=unfreeze_n
    ).to(device)
    
    total_p, train_p = count_params(model)
    print(f'Trainable: {train_p:,} / {total_p:,} ({train_p/total_p*100:.2f}%)')
    
    criterion = nn.CrossEntropyLoss()
    
    # Discriminative learning rates: backbone gets lower LR
    if unfreeze_n > 0:
        backbone_params = []
        for i in range(len(backbone.blocks) - unfreeze_n, len(backbone.blocks)):
            backbone_params.extend(list(model.backbone.blocks[i].parameters()))
        backbone_params.extend(list(model.backbone.norm.parameters()))
        
        optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': 1e-5},          # backbone: very low LR
            {'params': model.classifier.parameters(), 'lr': 1e-3},  # head: normal LR
        ], weight_decay=1e-4)
        print(f'LR: backbone=1e-5, head=1e-3')
    else:
        optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
        print(f'LR: head=1e-3 (backbone frozen)')
    
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6
    )
    
    # Train
    best_train_acc = 0.0
    patience_counter = 0
    save_name = f'best_model_unfreeze{unfreeze_n}.pth'
    
    print(f'\n{"Epoch":>6} | {"Loss":>8} | {"Acc":>8}')
    print('-' * 30)
    
    for epoch in range(1, num_epochs + 1):
        loss, acc = train_one_epoch(model, clean_loader, criterion, optimizer, device)
        scheduler.step()
        
        if acc > best_train_acc:
            best_train_acc = acc
            torch.save(model.state_dict(), save_name)
            patience_counter = 0
        else:
            patience_counter += 1
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'{epoch:>6} | {loss:>8.4f} | {acc:>8.4f}')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break
    
    print(f'Best train acc: {best_train_acc:.4f}')
    
    # Load best and evaluate with threshold sweep
    model.load_state_dict(torch.load(save_name, map_location=device))
    
    best_threshold = 0.5
    best_macro_f1 = 0.0
    
    for threshold in np.arange(0.40, 0.95, 0.05):
        preds, labels, confs = predict_with_threshold(
            model, test_loader, threshold, clean_class_names, all_class_names, device
        )
        mf1 = f1_score(labels, preds, average='macro')
        if mf1 > best_macro_f1:
            best_macro_f1 = mf1
            best_threshold = threshold
    
    # Final evaluation at best threshold
    preds, labels, confs = predict_with_threshold(
        model, test_loader, best_threshold, clean_class_names, all_class_names, device
    )
    accuracy = (preds == labels).mean()
    
    print(f'\nBest threshold: {best_threshold:.2f}')
    print(f'Macro F1: {best_macro_f1:.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    print(classification_report(labels, preds, target_names=all_class_names, digits=4))
    
    return {
        'unfreeze_n': unfreeze_n,
        'trainable_params': train_p,
        'best_train_acc': best_train_acc,
        'threshold': best_threshold,
        'macro_f1': best_macro_f1,
        'accuracy': accuracy,
        'preds': preds,
        'labels': labels,
        'confs': confs,
        'model_path': save_name,
    }

In [ ]:
# Run experiments for 0, 1, 2, 3 unfrozen blocks
results = []

for n_unfreeze in [0, 1, 2, 3]:
    result = run_experiment(
        unfreeze_n=n_unfreeze,
        backbone=dinov2_backbone,
        clean_loader=clean_train_loader,
        test_loader=full_test_loader,
        clean_class_names=CLEAN_CLASS_NAMES,
        all_class_names=CLASS_NAMES,
        device=device,
    )
    results.append(result)
    
    # Free GPU memory between experiments
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. Results Comparison

In [ ]:
# Build comparison table
comp_df = pd.DataFrame([{
    'Unfrozen Blocks': r['unfreeze_n'],
    'Trainable Params': f"{r['trainable_params']:,}",
    'Train Acc': f"{r['best_train_acc']:.4f}",
    'Threshold': f"{r['threshold']:.2f}",
    'Macro F1': f"{r['macro_f1']:.4f}",
    'Accuracy': f"{r['accuracy']:.4f}",
} for r in results])

print('=' * 90)
print('UNFREEZE LEVEL COMPARISON')
print('=' * 90)
print(comp_df.to_string(index=False))
print('=' * 90)

best_result = max(results, key=lambda r: r['macro_f1'])
print(f'\nBest: {best_result["unfreeze_n"]} unfrozen blocks '
      f'(Macro F1: {best_result["macro_f1"]:.4f}, threshold: {best_result["threshold"]:.2f})')

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

unfreeze_vals = [r['unfreeze_n'] for r in results]
f1_vals = [r['macro_f1'] for r in results]
acc_vals = [r['accuracy'] for r in results]
param_vals = [r['trainable_params'] for r in results]

# F1 and Accuracy
x = np.arange(len(unfreeze_vals))
w = 0.35
axes[0].bar(x - w/2, f1_vals, w, label='Macro F1', color='#4CAF50')
axes[0].bar(x + w/2, acc_vals, w, label='Accuracy', color='#2196F3')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{n} blocks' for n in unfreeze_vals])
axes[0].set_ylabel('Score')
axes[0].set_title('Performance by Unfreeze Level', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0.7, 1.0)
axes[0].grid(axis='y', alpha=0.3)

for i, (f1v, accv) in enumerate(zip(f1_vals, acc_vals)):
    axes[0].text(i - w/2, f1v + 0.005, f'{f1v:.4f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, accv + 0.005, f'{accv:.4f}', ha='center', fontsize=8)

# Trainable params vs F1
axes[1].plot(param_vals, f1_vals, 'go-', markersize=10, linewidth=2)
for i, n in enumerate(unfreeze_vals):
    axes[1].annotate(f'{n} blocks', (param_vals[i], f1_vals[i]),
                     textcoords='offset points', xytext=(10, 5), fontsize=9)
axes[1].set_xlabel('Trainable Parameters')
axes[1].set_ylabel('Macro F1')
axes[1].set_title('Params vs Performance', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('unfreeze_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Best Model — Detailed Evaluation

In [ ]:
best = best_result
preds = best['preds']
labels = best['labels']

print(f'Best model: {best["unfreeze_n"]} unfrozen blocks, threshold={best["threshold"]:.2f}')
print(f'Macro F1:  {best["macro_f1"]:.4f}')
print(f'Accuracy:  {best["accuracy"]:.4f}')
print(f'\n{"="*70}')
print('CLASSIFICATION REPORT')
print(f'{"="*70}')
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Recall)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_best_unfreeze.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confidence distribution
confs = best['confs']
threshold = best['threshold']

fig, ax = plt.subplots(figsize=(10, 5))

other_idx = CLASS_NAMES.index('other')
other_mask = labels == other_idx
defined_mask = ~other_mask

ax.hist(confs[defined_mask], bins=20, alpha=0.7, label='Defined classes', color='steelblue')
ax.hist(confs[other_mask], bins=20, alpha=0.7, label='Other', color='tomato')
ax.axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.2f}')
ax.set_xlabel('Max Softmax Confidence')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution: Defined Classes vs Other', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Defined classes — mean conf: {confs[defined_mask].mean():.3f}, min: {confs[defined_mask].min():.3f}')
print(f'Other class    — mean conf: {confs[other_mask].mean():.3f}, max: {confs[other_mask].max():.3f}')

## 8. Full Project Comparison

In [ ]:
full_comparison = pd.DataFrame({
    'Experiment': [
        'EfficientNet-B0 + imbalance mitigation',
        'EfficientNet-B0 baseline',
        'DINOv2 6-class (before relabeling)',
        'DINOv2 + CutMix/MixUp',
        'DINOv2 6-class (after relabeling)',
        'DINOv2 frozen + threshold',
        f'DINOv2 unfreeze-{best["unfreeze_n"]} + threshold',
    ],
    'Macro F1': [
        0.7645, 0.8089, 0.8244, 0.7896,
        0.8261, 0.8628, best['macro_f1'],
    ],
    'Accuracy': [
        0.7667, 0.7833, 0.8167, 0.7833,
        0.8333, 0.8500, best['accuracy'],
    ],
    'Notes': [
        'Original test labels',
        'Original test labels',
        'Original test labels',
        'Original test labels',
        'Relabeled test',
        'Relabeled test',
        'Relabeled test',
    ]
})

print('=' * 95)
print('FULL PROJECT COMPARISON')
print('=' * 95)
print(full_comparison.to_string(index=False))
print('=' * 95)

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(12, 6))
y = np.arange(len(full_comparison))
w = 0.35

ax.barh(y + w/2, full_comparison['Macro F1'], w, label='Macro F1', color='#4CAF50')
ax.barh(y - w/2, full_comparison['Accuracy'], w, label='Accuracy', color='#2196F3')

ax.set_yticks(y)
ax.set_yticklabels(full_comparison['Experiment'], fontsize=8)
ax.set_xlabel('Score')
ax.set_title('Full Project — Experiment Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0.6, 1.0)
ax.grid(axis='x', alpha=0.3)

for i, (f1v, accv) in enumerate(zip(full_comparison['Macro F1'], full_comparison['Accuracy'])):
    ax.text(f1v + 0.003, i + w/2, f'{f1v:.4f}', va='center', fontsize=7)
    ax.text(accv + 0.003, i - w/2, f'{accv:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('full_project_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

### Partial Unfreezing Results:
- Compared 0 (frozen baseline), 1, 2, 3 unfrozen transformer blocks
- Each experiment uses discriminative learning rates: backbone at 1e-5, head at 1e-3
- All experiments use 5-class training + confidence thresholding for `other`

### Key Question Answered:
Does domain-adapting DINOv2's high-level features improve classification on our small dataset?

Check the comparison table above — if unfreezing helps, you'll see higher Macro F1 with 1-2 blocks unfrozen. If it hurts, the frozen baseline (0 blocks) wins, meaning the features are already good enough and the extra parameters just overfit.